# 7차시 · RAG 직접 만들어 보기 🦘

**LangChain + HuggingFace + FAISS**

이 노트북은 강의에서 배운 **RAG(Retrieval-Augmented Generation, 검색 증강 생성)** 를
직접 코드로 따라 만들어 보는 실습입니다.

> **한 줄 요약** — RAG = **검색(Retrieval)** + **생성(Generation)**
> 질문과 관련된 문서를 먼저 *찾아서*, LLM에게 함께 건네주면 → 모르는 것도 근거 있게 답합니다.

우리가 만들 순서 (강의 파이프라인 그대로):

| 단계 | 하는 일 | 사용하는 도구 |
|---|---|---|
| 1 | 문서 준비 | (예제 텍스트) |
| 2 | 청크 분할 | `RecursiveCharacterTextSplitter` |
| 3 | 임베딩 | HuggingFace **BGE** (`bge-small-en-v1.5`) |
| 4 | 벡터 저장 | **FAISS** |
| 5 | 검색기 만들기 | `retriever` |
| 6 | LLM 준비 | HuggingFace `flan-t5-base` |
| 7 | 검색+생성 함수 | 직접 만드는 `ask()` |
| 8 | 질문하기 | `ask("...")` |

맨 아래에는 **실습하기** 코너가 있습니다. 먼저 위를 그대로 실행한 뒤,
설정값을 바꿔가며 무엇이 달라지는지 관찰합니다.


## 0단계 · 라이브러리 설치 ⚙️

아래 셀을 실행해 필요한 라이브러리를 설치합니다. (한 번만 하면 됩니다.)

- `langchain`, `langchain-community`, `langchain-huggingface`, `langchain-text-splitters` : LangChain 본체와 부품들
- `faiss-cpu` : 벡터 검색 라이브러리 FAISS (CPU 버전)
- `sentence-transformers` : 임베딩 모델을 돌리는 도구
- `transformers` : HuggingFace LLM을 돌리는 도구

> 💡 **모든 모델은 무료·공개 모델**이라 API 키가 필요 없습니다.
> 대신 처음 실행할 때 모델을 내려받느라 몇 분 걸릴 수 있어요. GPU도 필요 없습니다.

> ⚠️ **Colab 안내** — 설치 후 아래 '불러오기' 셀에서 오류가 나면,
> 상단 메뉴 **런타임 → 세션 다시 시작**을 누르고 **설치 셀은 건너뛴 채** 불러오기부터 다시 실행하세요.
> (설치 과정에서 일부 패키지가 갱신되면 한 번 재시작이 필요할 수 있습니다.)


In [1]:
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters
!pip install -q faiss-cpu sentence-transformers transformers pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 17.5 MB/s eta 0:00:00


## 라이브러리 불러오기 📦

설치한 도구들을 파이썬으로 불러옵니다. 각 줄이 무엇을 가져오는지 주석으로 적어 두었습니다.


In [2]:
import warnings
warnings.filterwarnings("ignore")  # 경고 메시지를 숨겨 결과를 깔끔하게

# 문서를 조각으로 자르는 도구
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 텍스트를 벡터로 바꾸는 임베딩 모델
from langchain_huggingface import HuggingFaceEmbeddings
# 벡터를 저장하고 검색하는 FAISS
from langchain_community.vectorstores import FAISS
# HuggingFace LLM(생성 모델)을 불러오는 도구
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("불러오기 완료!")

불러오기 완료!


## 1단계 · 문서 준비 📄

RAG는 "우리가 가진 문서"에서 답을 찾습니다. 실제로는 PDF나 웹페이지를 불러오지만,
여기서는 바로 실행되도록 **예제 텍스트**를 노트북 안에 넣어 두었습니다.
(내용은 오늘 배운 RAG·FAISS·LangChain 설명이라, 답이 맞았는지 눈으로 확인하기 좋아요.)

> 🔎 **참고** — 실제 PDF를 쓰고 싶다면:
> ```python
> from langchain_community.document_loaders import PyPDFLoader
> documents = PyPDFLoader("파일이름.pdf").load()
> ```
> 이렇게 불러온 뒤 아래 2단계로 이어가면 됩니다.

> 📝 예제 문서와 질문은 모델(영어 기반)이 잘 답하도록 **영어**로 준비했습니다.


In [18]:
knowledge = """
Retrieval-Augmented Generation, or RAG, is a technique that combines information
retrieval with text generation. Instead of relying only on what a language model
memorized during training, a RAG system first searches a collection of documents
for relevant passages and then gives those passages to the model as context.
This reduces hallucination and lets the model answer questions about private or
up-to-date information.

A RAG pipeline has two phases. In the indexing phase, documents are loaded, split
into small chunks, converted into embedding vectors, and stored in a vector
database. In the query phase, the user question is turned into a vector, the most
similar chunks are retrieved, and those chunks are passed to the language model to
generate a grounded answer.

FAISS stands for Facebook AI Similarity Search. It is an open-source library that
stores embedding vectors and searches for the most similar ones very quickly.
FAISS can handle millions of vectors and is one of the most widely used tools for
similarity search.

An embedding is a list of numbers that represents the meaning of a piece of text.
When two texts have a similar meaning, their embedding vectors are close to each
other. BGE (bge-small-en-v1.5) is a small and fast model that creates sentence
embeddings.

LangChain is a framework for building applications powered by large language
models. It provides building blocks such as document loaders, text splitters,
embeddings, vector stores, retrievers, and chains, so developers can connect these
pieces together with only a few lines of code.
"""

print("문서 길이:", len(knowledge), "글자")

문서 길이: 1580 글자


## 2단계 · 청크(조각) 분할 ✂️

긴 문서를 통째로 쓰면 검색이 뭉툭해집니다. 그래서 **읽기 좋은 작은 조각(chunk)** 으로 자릅니다.

**`RecursiveCharacterTextSplitter`의 매개변수**

- **`chunk_size`** : 조각 하나에 담을 **최대 글자 수**.
  → 크면 조각이 길고 개수는 줄어듭니다. 작으면 조각이 잘게 나뉩니다.
- **`chunk_overlap`** : 이웃한 조각끼리 **겹쳐서** 가져올 글자 수.
  → 조각 경계에서 문맥(문장)이 끊기지 않도록 살짝 겹쳐 줍니다.

**`create_documents`의 매개변수**

- **첫 번째 인자(리스트)** : 자를 텍스트들의 목록. 여기서는 `[knowledge]` 하나만 넣습니다.


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,     # 조각 하나의 최대 글자 수
    chunk_overlap=50,   # 조각끼리 겹치는 글자 수
)

chunks = splitter.create_documents([knowledge])

print("만들어진 조각 개수:", len(chunks))
print("\n첫 번째 조각 미리보기:")
print(chunks[0].page_content)

만들어진 조각 개수: 7

첫 번째 조각 미리보기:
Retrieval-Augmented Generation, or RAG, is a technique that combines information
retrieval with text generation. Instead of relying only on what a language model
memorized during training, a RAG system first searches a collection of documents


## 3단계 · 임베딩 모델 준비 🔢

**임베딩(embedding)** 은 텍스트를 *의미를 담은 숫자 벡터* 로 바꾸는 것입니다.
의미가 비슷하면 벡터도 가까워지고, 그 덕분에 "비슷한 문서 찾기"가 가능해집니다.

이번에는 HuggingFace의 **BGE** 임베딩 모델을 사용합니다.
BGE는 검색 성능이 좋아 실무에서 널리 쓰이는 모델입니다. (BAAI에서 공개, API 키 불필요)

**`HuggingFaceEmbeddings`의 매개변수**

- **`model_name`** : 사용할 임베딩 모델 이름.
  → `BAAI/bge-small-en-v1.5` 는 가볍고 빠른 BGE 모델입니다. (더 정확하게 하려면 `bge-base-en-v1.5`)
- **`encode_kwargs`** : 벡터를 만들 때의 추가 옵션(딕셔너리).
  - **`"normalize_embeddings"`** : `True`면 벡터 길이를 1로 맞춰 줍니다.
    → BGE는 이 정규화를 켜야 유사도(코사인) 비교가 잘 됩니다. **BGE에서는 True 권장!**


In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",        # HuggingFace BGE 임베딩 모델
    encode_kwargs={"normalize_embeddings": True} # BGE는 벡터 정규화를 켭니다
)

print("BGE 임베딩 모델 준비 완료!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

BGE 임베딩 모델 준비 완료!


## 4단계 · FAISS 벡터 저장소 만들기 🗄️

이제 조각들을 벡터로 바꿔 **FAISS**에 저장합니다.
아래 한 줄이 두 가지를 한 번에 처리합니다 → ① 모든 조각을 임베딩으로 변환 ② FAISS 인덱스 생성.

**`FAISS.from_documents`의 매개변수**

- **첫 번째 인자 `documents`** : 저장할 문서 조각들 (2단계에서 만든 `chunks`).
- **두 번째 인자 `embedding`** : 텍스트를 벡터로 바꿀 임베딩 모델 (3단계의 `embeddings`).


In [6]:
vectorstore = FAISS.from_documents(chunks, embeddings)

print("FAISS 벡터 저장소 완성! 저장된 조각 수:", vectorstore.index.ntotal)

FAISS 벡터 저장소 완성! 저장된 조각 수: 7


### (잠깐) 검색이 되는지 확인해 보기 🔎

LLM을 붙이기 전에, FAISS가 질문과 비슷한 조각을 잘 찾아오는지 먼저 확인해 봅시다.

**`similarity_search`의 매개변수**

- **`query`** : 찾고 싶은 질문/문장.
- **`k`** : 가장 비슷한 조각을 **몇 개** 가져올지.


In [7]:
found = vectorstore.similarity_search(query="What is FAISS?", k=2)

for i, doc in enumerate(found):
    print(f"--- 검색된 조각 {i} ---")
    print(doc.page_content, "\n")

--- 검색된 조각 0 ---
FAISS stands for Facebook AI Similarity Search. It is an open-source library that
stores embedding vectors and searches for the most similar ones very quickly.
FAISS can handle millions of vectors and is one of the most widely used tools for
similarity search. 

--- 검색된 조각 1 ---
Retrieval-Augmented Generation, or RAG, is a technique that combines information
retrieval with text generation. Instead of relying only on what a language model
memorized during training, a RAG system first searches a collection of documents 



## 5단계 · 검색기(retriever) 만들기 🔍

방금 쓴 FAISS를 **검색기(retriever)** 로 감싸 둡니다.
그러면 이후에 "질문 → 관련 조각"을 자동으로 꺼내올 수 있습니다.

**`as_retriever`의 매개변수**

- **`search_kwargs`** : 검색 옵션을 담은 딕셔너리.
  - **`"k"`** : 질문마다 가장 비슷한 조각을 **몇 개** 가져올지. (여기서는 2개)


In [8]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

print("검색기 준비 완료!")

검색기 준비 완료!


## 6단계 · LLM(생성 모델) 준비 🤖

검색된 조각을 읽고 **답을 써 줄** 언어 모델을 준비합니다.
강의와 동일하게 공개 모델 `google/flan-t5-base` 를 사용합니다. (처음 실행 시 다운로드에 시간이 걸려요.)

모델을 불러오고, **질문을 넣으면 답을 돌려주는** 작은 함수 `generate()` 를 직접 만듭니다.

**모델 불러오기**

- **`AutoTokenizer.from_pretrained(model_name)`** : 이 모델 전용 **토크나이저**(문장↔숫자 변환기)를 불러옵니다.
- **`AutoModelForSeq2SeqLM.from_pretrained(model_name)`** : 문장을 입력받아 새 문장을 만드는 **생성 모델**을 불러옵니다.

**`generate()` 안에서 쓰는 함수의 매개변수**

- **`tokenizer(...)`**
  - **`return_tensors="pt"`** : 결과를 PyTorch 텐서로 반환.
  - **`truncation=True`, `max_length=512`** : 입력이 너무 길면 512 토큰까지만 자릅니다(오류 방지).
- **`model.generate(...)`**
  - **`max_new_tokens=256`** : 답변으로 **새로 생성할 최대 토큰 수**.
- **`tokenizer.decode(...)`**
  - **`skip_special_tokens=True`** : 결과에서 특수 토큰을 빼고 사람이 읽는 문장으로 바꿉니다.

> 💡 강의 슬라이드에서는 `pipeline`으로 감쌌지만, 최신 `transformers`에서는 그 방식이 바뀌어
> 여기서는 모델을 **직접 호출**합니다. 결과는 같고, 오히려 동작(숫자로 바꾸기 → 생성 → 다시 문장)이 더 잘 보입니다.


In [9]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate(prompt):
    # 1) 문장을 숫자로 (너무 길면 512 토큰까지만 사용)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    # 2) 모델이 답을 생성
    outputs = model.generate(**inputs, max_new_tokens=256)
    # 3) 숫자를 다시 문장으로
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("LLM 준비 완료!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM 준비 완료!


## 7단계 · 검색 + 생성 함수 만들기 🔗

이제 **검색기 + LLM**을 이어 붙여 질문에 답하는 함수를 만듭니다. 이 흐름이 바로 RAG입니다.
복잡한 도구 없이 **세 줄**로 직접 만들어 보면 동작이 한눈에 보입니다.

**`ask()` 함수 안에서 일어나는 일**

- **① 검색** : `retriever.invoke(question)` → 질문과 비슷한 조각들을 가져옵니다.
- **② 합치기** : 가져온 조각들을 질문과 함께 하나의 프롬프트로 만듭니다.
- **③ 생성** : `generate(prompt)` → 6단계에서 만든 함수가 그 내용을 근거로 답을 씁니다.

**함수의 매개변수**

- **`question`** : 사용자가 묻는 질문 문장.


In [10]:
def ask(question):
    docs = retriever.invoke(question)                       # ① 관련 조각 검색
    context = "\n\n".join(d.page_content for d in docs)     # ② 조각들을 이어 붙이기
    prompt = (
        "Answer the question using the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )
    answer = generate(prompt)                              # ③ LLM이 답 생성
    return answer, docs

print("ask() 함수 준비 완료! 이제 질문할 수 있어요.")

ask() 함수 준비 완료! 이제 질문할 수 있어요.


## 8단계 · 질문하기 🙋

드디어 질문을 던져 봅니다. `ask()`는 **답변**과 **근거 조각**을 함께 돌려줍니다.

**`ask`의 매개변수**

- **첫 번째 인자** : 질문 문장(문자열).

반환값:
- **`answer`** : LLM이 만든 답변
- **`docs`** : 답의 근거가 된 문서 조각들


In [11]:
answer, docs = ask("What does FAISS stand for?")

print("❓ 질문:", "What does FAISS stand for?")
print("💬 답변:", answer)
print("\n📚 근거가 된 조각:")
for doc in docs:
    print("-", doc.page_content[:80], "...")

❓ 질문: What does FAISS stand for?
💬 답변: Facebook AI Similarity Search

📚 근거가 된 조각:
- FAISS stands for Facebook AI Similarity Search. It is an open-source library tha ...
- Retrieval-Augmented Generation, or RAG, is a technique that combines information ...


다른 질문도 던져 보세요. 예: `"What are the two phases of a RAG pipeline?"`, `"What is an embedding?"`


In [12]:
answer, docs = ask("What is an embedding?")
print("💬 답변:", answer)

💬 답변: a list of numbers that represents the meaning of a piece of text


---
# 🧪 실습하기

여기까지 **그대로 실행**했다면 RAG가 완성된 것입니다! 👏
이제 **설정값을 바꿔 가며** 무엇이 달라지는지 직접 관찰해 봅시다.

먼저, 위 2~7단계를 **한 번에** 만들어 주는 도우미 함수를 준비합니다.
(매번 셀을 다시 실행하지 않아도 되도록 묶은 것뿐입니다.)


In [13]:
def build_rag(chunk_size, chunk_overlap, k=2):
    # 2단계: 분할
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = splitter.create_documents([knowledge])

    # 3~5단계: 임베딩 → FAISS → 검색기
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})

    # 7단계: 이 설정으로 질문에 답하는 함수 만들기
    def ask_fn(question):
        docs = retriever.invoke(question)
        context = "\n\n".join(d.page_content for d in docs)
        prompt = (
            "Answer the question using the context below.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {question}\nAnswer:"
        )
        return generate(prompt)

    print(f"chunk_size={chunk_size}, chunk_overlap={chunk_overlap} → 조각 {len(chunks)}개")
    return ask_fn

## 실습 1 · chunk_size 바꿔 보기

`chunk_size`(조각 크기)를 바꾸면 무엇이 달라질까요?
아래 세 가지를 실행하고 **조각 개수**와 **답변**을 비교해 보세요.

👀 **관찰 포인트**
- `chunk_size`가 작아지면 조각 개수는 어떻게 되나요?
- 조각이 너무 작으면(예: 80) 하나의 조각에 정보가 충분히 담기지 못해 답이 부실해질 수 있습니다.
- 조각이 너무 크면(예: 800) 검색이 뭉툭해져 엉뚱한 부분까지 딸려올 수 있습니다.


In [19]:
question = "What does FAISS stand for?"

for size in [80, 300, 800]:
    qa_test = build_rag(chunk_size=size, chunk_overlap=0)
    print("   답변:", qa_test(question))
    print("-" * 50)

chunk_size=80, chunk_overlap=0 → 조각 34개
   답변: Facebook AI Similarity Search
--------------------------------------------------
chunk_size=300, chunk_overlap=0 → 조각 7개
   답변: Facebook AI Similarity Search
--------------------------------------------------
chunk_size=800, chunk_overlap=0 → 조각 3개
   답변: Facebook AI Similarity Search
--------------------------------------------------


## 실습 2 · overlap(겹침) 눈으로 확인하기

`chunk_overlap`은 조각끼리 **겹치는 글자 수**입니다.
겹침이 있으면 조각 경계에서 문장이 끊겨도 다음 조각이 앞부분을 다시 담아 **문맥이 이어집니다.**

아래 함수로 겹침이 **0일 때**와 **있을 때** 조각이 어떻게 나뉘는지 직접 눈으로 비교해 보세요.


In [21]:
def show_chunks(chunk_size, chunk_overlap):
    sp = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    pieces = sp.split_text(knowledge)
    print(f"[chunk_size={chunk_size}, overlap={chunk_overlap}] → 조각 {len(pieces)}개")
    for i, c in enumerate(pieces[:6]):
        print(f"\n--- 조각 {i} ---")
        print(c)
    print("=" * 60)

# 겹침 없음 vs 겹침 있음 비교
show_chunks(chunk_size=60, chunk_overlap=0)
show_chunks(chunk_size=60, chunk_overlap=30)

[chunk_size=60, overlap=0] → 조각 41개

--- 조각 0 ---
Retrieval-Augmented Generation, or RAG, is a technique that

--- 조각 1 ---
combines information

--- 조각 2 ---
retrieval with text generation. Instead of relying only on

--- 조각 3 ---
what a language model

--- 조각 4 ---
memorized during training, a RAG system first searches a

--- 조각 5 ---
collection of documents
[chunk_size=60, overlap=30] → 조각 41개

--- 조각 0 ---
Retrieval-Augmented Generation, or RAG, is a technique that

--- 조각 1 ---
or RAG, is a technique that combines information

--- 조각 2 ---
retrieval with text generation. Instead of relying only on

--- 조각 3 ---
Instead of relying only on what a language model

--- 조각 4 ---
memorized during training, a RAG system first searches a

--- 조각 5 ---
a RAG system first searches a collection of documents


> 🔍 겹침이 40일 때는 **어떤 조각의 끝부분이 다음 조각의 앞부분에 다시 등장**합니다(겹침).
> 겹침이 0일 때와 비교하면, 겹침이 있을수록 경계에서 문맥이 덜 끊기는 대신 조각 수는 늘어납니다.


## 실습 3 · 나만의 값으로 실험하기 ✍️

이제 직접 값을 바꿔 보세요. 아래 셀에서 숫자만 고쳐 실행하면 됩니다.

시도해 볼 만한 조합:
- `build_rag(chunk_size=200, chunk_overlap=0)`
- `build_rag(chunk_size=200, chunk_overlap=60)`
- `build_rag(chunk_size=500, chunk_overlap=100)`

그리고 같은 질문에 답이 어떻게 달라지는지 비교해 보세요.


In [22]:
question = "What are the two phases of a RAG pipeline, and what happens in each?"

qa_no_overlap = build_rag(chunk_size=150, chunk_overlap=0)
print("겹침 0:", qa_no_overlap(question))
print("-" * 50)

qa_overlap = build_rag(chunk_size=150, chunk_overlap=100)
print("겹침 100:", qa_overlap(question))

chunk_size=150, chunk_overlap=0 → 조각 18개
겹침 0: documents are loaded, split Retrieval-Augmented Generation
--------------------------------------------------
chunk_size=150, chunk_overlap=100 → 조각 18개
겹침 100: documents are loaded, split Retrieval-Augmented Generation


In [24]:
qa_more_context = build_rag(chunk_size=150, chunk_overlap=100, k=4)
print(qa_more_context(question))

chunk_size=150, chunk_overlap=100 → 조각 18개
documents are loaded, split Retrieval-Augmented Generation


## 정리 & 생각해볼 질문 🧠

오늘 우리는 **로드 → 분할 → 임베딩 → FAISS 저장 → 검색 → 생성** 순서로 RAG를 직접 만들었습니다.

스스로 답해 보세요:
1. `chunk_size`를 아주 작게(예: 50) 하면 답변 품질이 왜 나빠질까요?
2. `chunk_overlap`은 어떤 상황에서 특히 도움이 될까요?
3. `k`(가져오는 조각 수)를 늘리면 어떤 장점과 단점이 있을까요?

> **핵심** — RAG의 성능은 "얼마나 좋은 조각을, 얼마나 잘 찾아오는가"에 크게 좌우됩니다.
> 그래서 `chunk_size`·`overlap`·`k` 같은 값을 조절하는 것이 실무에서 매우 중요합니다. 🦘


1. 조각이 너무 작으면 한 조각 안에 완전한 의미 단위(문장)가 다 안 들어갑니다. 생성단계에서 LLM이 그 반쪽짜리 정보만 보고 답을 만들어야 해서, 답이 부정확하거나 불완전해집니다.

2. 문장이나 중요한 정보가 chunk 경계에 걸쳐 있을 때 도움이 됩니다

3.
장점: 더 많은 문맥(정보)을 LLM에게 줄 수 있어서, 답이 여러 조각에 걸쳐 있는 복합적인 질문(예: "두 phase가 뭐고 각각 뭘 하나요?")에 답할 확률이 올라갑니다.

단점: 관련 없는 조각까지 섞여 들어가면 오히려 LLM이 헷갈려서 답이 부정확해질 수 있고 프롬프트가 길어지면서 처리 속도가 느려지고, 특히 우리가 쓴 것 같은 작은 모델은 긴 컨텍스트를 잘 못 다뤄서 오히려 답변 품질이 떨어질 수도 있습니다.